In [ ]:
import sys; sys.path.append('..')
import sheet_convergence, sim_utils
import MeshFEM
from tri_mesh_viewer import TriMeshViewer
import numpy as np

import meshing, mesh, elastic_solid, energy

pts, _ = sheet_convergence.stripBoundary(1)
m = mesh.Mesh(*meshing.tetrahedralize_extruded_polylines([np.array(pts + [pts[0]])], [], thickness=4.0, maxVol=0.001), degree=1)
es = elastic_solid.ElasticSolid(m, energy.NeoHookeanYoungPoisson(3, 1, 0))

es_re = elastic_solid.ElasticSolidRotExtrap(m, energy.NeoHookeanYoungPoisson(3, 1, 0))

In [ ]:
v = TriMeshViewer(es, wireframe=True)
v.setCameraParams(((3.6996509275435927, 1.3705005944233914, 4.1946387036007655),
 (-0.18400815367026574, 0.9706515767088857, -0.15484352106373026),
 (0.0, 0.0, 0.0)))
v.show()

In [ ]:
x_rest = es.getVars().copy()
x = x_rest.copy()

In [ ]:
minZVars = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MIN_Z)
maxZVars = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MAX_Z)

x[minZVars[2::3]] *= 1.05
x[maxZVars[2::3]] *= 1.05

es.setVars(x)

es.computeEquilibrium([], fixedVars=(minZVars + maxZVars))
v.update()

In [ ]:
x_orig = es.getVars().copy()

In [ ]:
import compute_vibrational_modes
#lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(es, n=12, mtype=compute_vibrational_modes.MassMatrixType.IDENTITY, sigma=-1e-11)
lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(es, n=12, mtype=compute_vibrational_modes.MassMatrixType.FULL, sigma=-1e-10)

In [ ]:
es_re.setVars(x_orig)
es_re.energy()

In [ ]:
es_re.updateParametrization()

In [ ]:
es_re.energy()

In [ ]:
v2 = TriMeshViewer(es_re, wireframe=True)
v2.show()

In [ ]:
v2.setCameraParams(v.getCameraParams())

In [ ]:
es_re.setVars(x_orig)
v2.update()

In [ ]:
es_re.setVars(x_orig + 1.0 * modes[:, 9])
v2.update()

In [ ]:
es.setVars(x_orig + 1.0 * modes[:, 5])
print(es.energy())

In [ ]:
es_re.method = es_re.method.ElementExtrapolation
print(es_re.energy())
v2.update()

In [ ]:
es_re.method = es_re.method.ModalWarping
print(es_re.energy())
v2.update()

In [ ]:
magnitudes = np.linspace(0, 0.2, 100)
numModes = modes.shape[1]
def ModeData():
    return [[] for i in range(numModes)]
energies = {'Naive': ModeData(), 'ElementExtrap': ModeData(), 'ModalWarp': ModeData()}
for i, mode in enumerate(modes.T):
    for mag in magnitudes:
        es.setVars(x_orig + mag * mode)
        energies['Naive'][i].append(es.energy())
        es_re.setVars(x_orig + mag * mode)
        es_re.method = es_re.method.ElementExtrapolation
        energies['ElementExtrap'][i].append(es_re.energy())
        es_re.method = es_re.method.ModalWarping
        energies['ModalWarp'][i].append(es_re.energy())

In [ ]:
stresses  = {k: [np.gradient(e, magnitudes, edge_order=2) for e in v] for k, v in energies.items()}
stiffness = {k: [np.gradient(s, magnitudes, edge_order=2) for s in v] for k, v in stresses.items()}

In [ ]:
lwidths = {'Naive': 4, 'ElementExtrap': 2, 'ModalWarp': 1}

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
plt.figure(figsize=(12, 12))
for i, mode in enumerate(modes.T):
    plt.subplot(numModes // 3, 3, i + 1)
    for k in energies:
        plt.plot(magnitudes, energies[k][i], label=k, lw=lwidths[k])
    plt.title(f'Mode {i}')
    plt.legend()
plt.suptitle('Energy')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 12))
for i, mode in enumerate(modes.T):
    plt.subplot(numModes // 3, 3, i + 1)
    for k in energies:
        plt.plot(magnitudes, stresses[k][i], label=k, lw=2)
    plt.title(f'Mode {i}')
    plt.legend()
plt.suptitle('Stress')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 12))
for i, mode in enumerate(modes.T):
    plt.subplot(numModes // 3, 3, i + 1)
    for k in energies:
        plt.plot(magnitudes, stiffness[k][i], label=k, lw=2)
    plt.title(f'Mode {i}')
    plt.legend()
plt.suptitle('Stiffness')
plt.tight_layout()
plt.show()

In [ ]:
es_re.setVars(x_orig)

In [ ]:
v = modes[:, 0]
eps = 1e-1
es_re.elasticSolid.setVars(x_orig + eps * v)
es_re.setVars(x_orig + eps * v)
x_extrap_plus = es_re.elasticSolid.getVars().copy()
es_re.elasticSolid.setVars(x_orig - eps * v)
es_re.setVars(x_orig - eps * v)
x_extrap_minus = es_re.elasticSolid.getVars().copy()
x_extrap_velocity = (x_extrap_plus - x_extrap_minus) / (2 * eps)

In [ ]:
v2.update()

In [ ]:
import mode_viewer

In [ ]:
mview = mode_viewer.ModeViewer(es, modes, lambdas, wireframe=False, numSteps=2)

In [ ]:
mview.show()

In [ ]:
mview.amplitude = 0.5

In [ ]:
mview_esre = mode_viewer.ModeViewer(es_re, modes, lambdas, wireframe=False)

In [ ]:
mview_esre.amplitude = 0.5

In [ ]:
es_re.method = es_re.method.ElementExtrapolation

In [ ]:
es_re.method = es_re.method.ModalWarping

In [ ]:
mview_esre.show()

In [ ]:
from PIL import ImageDraw
from PIL import ImageFont

In [ ]:
from PIL import ImageChops

In [ ]:
# def renderModeImages(v, modes, amplitude, width=None, height=None, normalCreaseAngle = np.pi / 4):
width, height = 2048, 2048
zFightOffset=5e-4
amplitude = 0.5
kwargs = {}
images = []
normalCreaseAngle = np.pi / 4
if width is not None: kwargs['width'] = width
if height is not None: kwargs['height'] = height
orender = v.offscreenRenderer(**kwargs)
zFightOffsetVec = zFightOffset * np.linalg.inv(orender.matView @ orender.meshes[0].matModel)[0:3, 2]
currVars = es.getVars()
vg_undefo = es.visualizationGeometry(normalCreaseAngle)
for mode in modes.T:
    es.setVars(currVars + amplitude * mode)
    vg_defo = es.visualizationGeometry(np.pi / 4)
    V =  (vg_defo[0] + zFightOffsetVec)[vg_defo[1].ravel()]
    N =  vg_defo[2]
    C = np.repeat([[0.6, 0.8, 1.0, 0.5]], 3 * len(vg_defo[1]),   axis=0)
    F = np.arange(len(V), dtype=np.uint32)
    orender.addMesh(V, F, N, C, makeDefault=False)
    orender.meshes[-1].matModel = orender.meshes[0].matModel.copy()
    orender.meshes[0].alpha = 1.0
    orender.meshes[0].lineWidth = 0.0
    orender.render()
    orender.removeMesh(-1)
    images.append(orender.image().resize((768, 768)))
# # Common crop
# bbox = 
# for i, l in enumerate(lambdas):
#     img = images[i]
#     draw = ImageDraw.Draw(img)
#     #draw.text((img.width / 2, 20), f"Mode {i} - λ = {l}", fill=(0, 0, 0), font=ImageFont.truetype('fonts/FreeSans.ttf', size=20), anchor="mm")
#     print(img.getbbox())

In [ ]:
img.getbbox?

In [ ]:
images[11]

In [ ]:
images[0].tell

In [ ]:
images[10]